# 02B COF 描述符：从结构到机器学习特征

> 🟢 **Level A · 必须掌握** | 完成标准：区分 composition、crystal、pore 与 chemistry descriptors，并知道 feature 必须匹配 target。


## COF 背景衔接：从化学问题选择特征

[背景图](https://raw.githubusercontent.com/Wanteen/COF-ML-Tutorial/main/assets/cof_background.jpg) 展示了不同构筑体系与孔道结构。改变连接单元可能同时改变孔道几何与孔壁化学。预测 CO₂ 吸附量时，应结合孔径、孔隙率、密度和组成等互补特征；仅凭元素比例无法区分拓扑或层间堆积。

温度与压力必须一致处理：可以比较固定条件下的吸附量，也可以将条件明确纳入模型输入。吸附容量、选择性和传输性质是不同的目标，应先定义问题再选择描述符。


## 1. 一个最简单的例子
对于一个 COF，我们可以记录：
- density = 0.55 g/cm³；
- pore diameter = 22 Å；
- void fraction = 0.72；
- N fraction = 0.08。

这 4 个数字就构成了一组最简单的材料表示。


In [ ]:
import pandas as pd
df=pd.DataFrame({
 'COF':['A','B','C'],
 'density':[0.55,0.78,0.61],
 'pore_A':[28.0,17.5,22.1],
 'void_fraction':[0.72,0.51,0.65],
 'N_fraction':[0.08,0.13,0.05]
})
df


## 2. COF 描述符可以分成几类？

### Composition / chemistry（组成与化学）
例如 C/N/O/F 含量、官能团、linkage 类型。

### Geometry（几何）
例如孔径、孔隙率、密度、比表面积。

### Structure / topology（结构与拓扑）
例如 topology、2D/3D、stacking、层间距离。

不同 target 需要的信息不同。预测吸附量和预测 band gap，不应该机械使用完全相同的一组 feature。


## 3. 为什么只看化学式不够？
两个 COF 可以元素组成很接近，但孔径、拓扑或 stacking 完全不同。

因此 `composition → property` 对某些问题有用，但它看不到很多 COF 的关键结构信息。


In [ ]:
!pip -q install pymatgen matminer
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
formulas=['C6H6','C6H4N2','C6H4O2']
tmp=pd.DataFrame({'composition':[Composition(x) for x in formulas]})
feat=ElementProperty.from_preset('magpie')
feat.set_n_jobs(1)  # Three teaching examples: avoid multiprocessing overhead
out=feat.featurize_dataframe(tmp,col_id='composition',ignore_errors=True)
print('自动生成的列数:',out.shape[1]-1)
display(out.iloc[:,:8])


上面的 matminer 只是演示“可以自动生成 composition descriptors”。**不要因为工具能生成很多列，就默认这些 feature 都适合你的问题。**

对 COF，pore size、surface area、void fraction、topology、functional-group position、stacking 等通常需要额外来源或专门计算。


## 本章术语表
- representation：表示，材料如何被模型表达；
- descriptor：描述符，把材料信息转成数值；
- composition descriptor：组成描述符；
- geometric descriptor：几何描述符；
- categorical feature：类别特征，例如 linkage='imine'。


## 补充：特征分层与构造


### COF 特征分层

| 层级 | 例子 | 来源 |
|---|---|---|
| Composition | C/N/O/F fraction | formula / CIF |
| Crystal | a,b,c,angles,volume,density | CIF |
| Pore | PLD,LCD,ASA,void fraction,pore volume | pore software |
| Chemistry | functional groups, RAC/local environment | structure + chemistry tools |
| Learned | graph embedding | GNN |

能生成很多列，不代表都应该进入模型。


In [ ]:
import pandas as pd
example=pd.DataFrame({'COF':['A','B','C'],'density':[0.55,0.78,0.61],'PLD_A':[12,7.5,10.1],'LCD_A':[18,11.2,15.4],'void_fraction':[0.72,0.51,0.65],'N_fraction':[0.08,0.13,0.05]})
example['LCD_PLD_ratio']=example['LCD_A']/example['PLD_A']
display(example)


### Feature engineering
有物理意义地组合已有特征也是特征构造，例如 `LCD/PLD`、heteroatom fraction。复杂不等于更好。


## Exercises
1. 为 CO₂ adsorption 选择 5 个你认为重要的 feature，并说明原因。
2. 为 band gap 再选 5 个 feature。比较两组是否相同。
3. 为什么只使用元素组成无法区分两个 stacking 不同的 2D COF？

### 本章最低要求
能够解释 descriptor 是“材料信息 → 数字”的桥梁，并知道 COF 的几何、化学和结构信息往往需要组合使用。


## 数据来源与扩展阅读
[Dataset contracts / 数据使用约定](../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)
